# Cluster count experiment — does k=100 beat k=20 for cluster-derived anchors?

## 1. Setup

In [1]:
import os, sys, copy, gc
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity as _cos

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.preprocess import preprocess_pipeline, AUDIO_FEATURES
from src.recommender import (
    score_songs_by_audio, get_relevant_clusters,
    model as _st_model,
)
from src.embeddings import create_playlist_embedding
from src.evaluate import precision_at_k
from src.test_set import (
    TEST_PLAYLISTS, HELD_OUT_PLAYLISTS, get_ground_truth_indices,
)

TOP_K = 10
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'data'))
TEXT_EMBED_PATH = os.path.join(DATA_DIR, 'text_embeddings.npy')

print('Loading catalogue and scaler / k=20 K-Means (existing main pipeline)...')
df, audio_features, scaler, kmeans20 = preprocess_pipeline(os.path.join(DATA_DIR, 'spotify_data.csv'))
print(f'  catalogue: {len(df):,} rows')

gc.collect()
text_embeddings = np.load(TEXT_EMBED_PATH, mmap_mode='r')
print(f'  text embeddings (mmap): {text_embeddings.shape}  dtype={text_embeddings.dtype}')
print(f'  dev playlists: {len(TEST_PLAYLISTS)}    held-out playlists: {len(HELD_OUT_PLAYLISTS)}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding vibe anchors...
Vibe anchors ready.
Loading catalogue and scaler / k=20 K-Means (existing main pipeline)...
  catalogue: 1,159,748 rows
  text embeddings (mmap): (1159748, 384)  dtype=float32
  dev playlists: 15    held-out playlists: 15


## 2. Fit a new K-Means with k=100 and re-label songs

In [2]:
K_NEW = 100
print(f'Fitting K-Means with k={K_NEW} on standardised audio matrix (1.16M rows x 12 dims)...')
kmeans100 = KMeans(n_clusters=K_NEW, random_state=42, n_init=10).fit(audio_features)
print(f'  done. Inertia = {kmeans100.inertia_:.0f}')

df['cluster'] = kmeans100.predict(audio_features)

cluster_sizes = df['cluster'].value_counts()
print(f'  cluster sizes: min={cluster_sizes.min():,}  median={int(cluster_sizes.median()):,}  max={cluster_sizes.max():,}')

Fitting K-Means with k=100 on standardised audio matrix (1.16M rows x 12 dims)...
  done. Inertia = 3554147
  cluster sizes: min=1,115  median=10,991  max=25,289


## 3. Build cluster-anchors (one anchor per cluster, same logic as Section 9.5)

In [3]:
def build_cluster_anchors(df, scaler, kmeans, top_genres_per_cluster=4):
    centroids_std = kmeans.cluster_centers_
    centroids_raw = scaler.inverse_transform(centroids_std)

    anchors = []
    for c in range(kmeans.n_clusters):
        members = df[df['cluster'] == c]
        if len(members) == 0:
            continue
        top_genres = members['genre'].value_counts().head(top_genres_per_cluster).index.tolist()
        feats = {f: float(centroids_raw[c, i]) for i, f in enumerate(AUDIO_FEATURES)}

        descriptors = []
        if feats['energy']           >= 0.7: descriptors.append('high-energy')
        elif feats['energy']         <= 0.3: descriptors.append('low-energy calm')
        if feats['danceability']     >= 0.7: descriptors.append('danceable')
        if feats['valence']          >= 0.7: descriptors.append('happy upbeat')
        elif feats['valence']        <= 0.3: descriptors.append('sad melancholic')
        if feats['acousticness']     >= 0.6: descriptors.append('acoustic')
        if feats['instrumentalness'] >= 0.5: descriptors.append('instrumental')
        if feats['speechiness']      >= 0.2: descriptors.append('vocal-heavy')
        if feats['tempo']            >= 140: descriptors.append('fast tempo')
        elif feats['tempo']          <= 80:  descriptors.append('slow tempo')

        description = ' '.join(descriptors + top_genres).strip() or ' '.join(top_genres)
        anchors.append({
            'description': description,
            'genres':      top_genres,
            'year_range':  None,
            'features':    feats,
            'cluster_id':  c,
        })
    return anchors


CLUSTER_ANCHORS_100 = build_cluster_anchors(df, scaler, kmeans100)
print(f'Built {len(CLUSTER_ANCHORS_100)} cluster-anchors (k={K_NEW}).')
print()
print('Sample of 10 anchor descriptions:')
for a in CLUSTER_ANCHORS_100[:10]:
    print(f"  cluster {a['cluster_id']:>3}: {a['description']}")

Built 100 cluster-anchors (k=100).

Sample of 10 anchor descriptions:
  cluster   0: acoustic opera cantopop samba show-tunes
  cluster   1: high-energy sad melancholic instrumental black-metal grindcore death-metal garage
  cluster   2: high-energy danceable happy upbeat instrumental deep-house breakbeat minimal-techno afrobeat
  cluster   3: high-energy happy upbeat forro party sertanejo power-pop
  cluster   4: high-energy sad melancholic heavy-metal death-metal goth black-metal
  cluster   5: low-energy calm sad melancholic acoustic instrumental sleep classical ambient piano
  cluster   6: high-energy danceable vocal-heavy dancehall hip-hop hardcore french
  cluster   7: happy upbeat sad sertanejo pop-film rock-n-roll
  cluster   8: acoustic vocal-heavy comedy show-tunes german rock-n-roll
  cluster   9: high-energy sad melancholic death-metal emo heavy-metal black-metal


## 4. Interpret playlist names against the new anchor set

In [4]:
_cluster_anchor_emb_100 = _st_model.encode(
    [a['description'] for a in CLUSTER_ANCHORS_100], show_progress_bar=False
)
print(f'Encoded {_cluster_anchor_emb_100.shape[0]} anchor descriptions to {_cluster_anchor_emb_100.shape[1]}-dim.')


def interpret_clusters_100(playlist_name, top_n_anchors=3, dominance_margin=0.025):
    vec = _st_model.encode([playlist_name])
    sims = _cos(vec, _cluster_anchor_emb_100)[0]
    sorted_idx = sims.argsort()[::-1]
    top_score    = sims[sorted_idx[0]]
    second_score = sims[sorted_idx[1]] if len(sorted_idx) > 1 else 0.0
    effective_n  = 1 if (top_score - second_score) >= dominance_margin else top_n_anchors

    top_indices = sorted_idx[:effective_n]
    top_scores  = sims[top_indices]
    weights     = np.exp(top_scores) / np.exp(top_scores).sum()

    blended = {f: 0.0 for f in AUDIO_FEATURES}
    for i in range(effective_n):
        for f in AUDIO_FEATURES:
            blended[f] += weights[i] * CLUSTER_ANCHORS_100[top_indices[i]]['features'][f]

    genre_scores = {}
    for i in range(effective_n):
        for g in CLUSTER_ANCHORS_100[top_indices[i]]['genres']:
            genre_scores[g] = genre_scores.get(g, 0) + weights[i]
    allowed_genres = [g for g, s in genre_scores.items() if s > 0.1]

    matched = [CLUSTER_ANCHORS_100[i]['description'] for i in top_indices]
    return {
        'features':         blended,
        'genres':           allowed_genres,
        'matched_anchors':  matched,
        'anchor_weights':   weights.tolist(),
        'year_range':       None,
    }


for name in ['Workout', 'Sad Songs', '2000s Synthpop', 'Spanish Holiday', 'Late Night Drive']:
    interp = interpret_clusters_100(name)
    print(f"{name!r:<25} -> {interp['matched_anchors'][0][:80]}")

Encoded 100 anchor descriptions to 384-dim.
'Workout'                 -> high-energy hardstyle alt-rock emo dance
'Sad Songs'               -> happy upbeat sad sertanejo pop-film rock-n-roll
'2000s Synthpop'          -> high-energy sad melancholic instrumental minimal-techno deep-house club techno
'Spanish Holiday'         -> happy upbeat salsa forro sertanejo rock-n-roll
'Late Night Drive'        -> sad melancholic instrumental slow tempo sleep grindcore classical ambient


## 5. Evaluate hybrid_full with k=100 cluster-anchors on dev + held-out

In [5]:
def hybrid_full_clusters_100(playlist_name, df, text_embeddings, top_k=10,
                              scaler=None, kmeans=None, **kw):
    interp = interpret_clusters_100(playlist_name)
    rel_clusters = (get_relevant_clusters(interp['features'], scaler, kmeans)
                    if scaler is not None and kmeans is not None else None)
    audio_scores = score_songs_by_audio(
        df, interp['features'], interp['genres'],
        rel_clusters, year_range=None,
    )
    text_vec = create_playlist_embedding(playlist_name)
    text_sim = _cos(text_vec, text_embeddings)[0]
    text_norm = (text_sim - text_sim.min()) / (text_sim.max() - text_sim.min() + 1e-8)
    final_scores = 0.6 * audio_scores + 0.4 * text_norm
    top = final_scores.argsort()[-top_k:][::-1]
    return df.iloc[top].assign(score=final_scores[top].round(3)), interp


def eval_on(playlists, label):
    precs = []
    for tp in playlists:
        recs, _ = hybrid_full_clusters_100(
            tp['name'], df, text_embeddings,
            top_k=TOP_K, scaler=scaler, kmeans=kmeans100,
        )
        relevant = list(get_ground_truth_indices(df, tp))
        precs.append(precision_at_k(recs.index.tolist(), relevant, TOP_K))
    mean_p = float(np.mean(precs))
    std_p  = float(np.std(precs))
    print(f'  {label:<10} mean P@10 = {mean_p:.4f}   std = {std_p:.4f}   per-playlist: {[round(p, 2) for p in precs]}')
    return mean_p, std_p


print('Evaluating hybrid_full with k=100 cluster-anchors...')
print()
dev_p100,    dev_std100    = eval_on(TEST_PLAYLISTS,     'dev')
held_p100,   held_std100   = eval_on(HELD_OUT_PLAYLISTS, 'held-out')

Evaluating hybrid_full with k=100 cluster-anchors...



c:\Users\janis\OneDrive\Documents\TSI Studies\Bachelor Thesis\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\janis\OneDrive\Documents\TSI Studies\Bachelor Thesis\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\janis\OneDrive\Documents\TSI Studies\Bachelor Thesis\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\janis\OneDrive\Documents\TSI Studies\Bachelor Thesis\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\janis\OneDrive\Documents\TSI Studies\Bachelor T

  dev        mean P@10 = 0.6600   std = 0.3738   per-playlist: [1.0, 0.2, 0.0, 1.0, 0.3, 0.2, 0.9, 1.0, 1.0, 1.0, 0.5, 1.0, 0.6, 0.2, 1.0]


c:\Users\janis\OneDrive\Documents\TSI Studies\Bachelor Thesis\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\janis\OneDrive\Documents\TSI Studies\Bachelor Thesis\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\janis\OneDrive\Documents\TSI Studies\Bachelor Thesis\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\janis\OneDrive\Documents\TSI Studies\Bachelor Thesis\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\janis\OneDrive\Documents\TSI Studies\Bachelor T

  held-out   mean P@10 = 0.2533   std = 0.3052   per-playlist: [0.0, 0.5, 0.6, 0.5, 0.0, 0.2, 0.7, 0.0, 0.0, 0.0, 0.0, 0.0, 0.9, 0.0, 0.4]


## 6. Compare to existing systems

In [6]:
HANDCRAFTED_DEV_P10  = 0.8067
HANDCRAFTED_HELD_P10 = 0.3400
CLUSTERS20_HELD_P10  = 0.24

summary = pd.DataFrame([
    {'system': 'hybrid_full (hand-crafted, 16 anchors)',     'dev P@10': HANDCRAFTED_DEV_P10,  'held-out P@10': HANDCRAFTED_HELD_P10},
    {'system': 'hybrid_full (cluster-derived, k=20, 20 anchors)',  'dev P@10': float('nan'),         'held-out P@10': CLUSTERS20_HELD_P10},
    {'system': 'hybrid_full (cluster-derived, k=100, 100 anchors)','dev P@10': dev_p100,             'held-out P@10': held_p100},
])
print(summary.to_string(index=False))
print()

delta_held = held_p100 - HANDCRAFTED_HELD_P10
delta_dev  = dev_p100  - HANDCRAFTED_DEV_P10
delta_k20  = held_p100 - CLUSTERS20_HELD_P10

print(f'Delta vs hand-crafted   (held-out): {delta_held:+.4f}')
print(f'Delta vs hand-crafted   (dev):      {delta_dev:+.4f}')
print(f'Delta vs cluster k=20  (held-out):  {delta_k20:+.4f}')
print()
if held_p100 > HANDCRAFTED_HELD_P10:
    print('=> k=100 cluster-anchors BEAT hand-crafted on held-out P@10.')
    print('   This would be a strong thesis result. Worth considering switching the main system.')
elif held_p100 > CLUSTERS20_HELD_P10:
    print('=> k=100 cluster-anchors are BETTER than k=20 cluster-anchors but still LOSE to hand-crafted.')
    print('   Useful for the thesis: shows the cluster-count knob helps, but hand-crafting still wins.')
else:
    print('=> k=100 cluster-anchors do NOT improve over k=20.')
    print('   Useful for the thesis: shows that adding cluster resolution alone does not solve the coverage problem.')

                                           system  dev P@10  held-out P@10
           hybrid_full (hand-crafted, 16 anchors)    0.8067       0.340000
  hybrid_full (cluster-derived, k=20, 20 anchors)       NaN       0.240000
hybrid_full (cluster-derived, k=100, 100 anchors)    0.6600       0.253333

Delta vs hand-crafted   (held-out): -0.0867
Delta vs hand-crafted   (dev):      -0.1467
Delta vs cluster k=20  (held-out):  +0.0133

=> k=100 cluster-anchors are BETTER than k=20 cluster-anchors but still LOSE to hand-crafted.
   Useful for the thesis: shows the cluster-count knob helps, but hand-crafting still wins.


## 7. Save artefacts for the thesis

In [7]:
out_csv = os.path.join(DATA_DIR, 'cluster_count_results.csv')
summary.to_csv(out_csv, index=False)
print(f'Saved comparison table: {out_csv}')
summary

Saved comparison table: c:\Users\janis\OneDrive\Documents\TSI Studies\Bachelor Thesis\data\cluster_count_results.csv


,system,dev P@10,held-out P@10
0,"hybrid_full (hand-crafted, 16 anchors)",0.8067,0.340000
1,"hybrid_full (cluster-derived, k=20, 20 anchors)",NaN,0.240000
2,"hybrid_full (cluster-derived, k=100, 100 anchors)",0.6600,0.253333
